<a href="https://colab.research.google.com/github/zackives/upenn-cis5450-hw/blob/main/19_Module_4_Part_I_Time_Varying.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time-Varying Data


## Basic Setup

## Visualizing Timeseries

In [ ]:
!wget -nc https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv

In [ ]:
!head daily-min-temperatures.csv

In [ ]:
import pandas as pd
from datetime import datetime

# Read as a dataframe, convert date entries to the index
data = pd.read_csv('daily-min-temperatures.csv', \
                   dtype={'Date': str, 'Temp': float})
data['date'] = data['Date'].apply(\
                                  lambda x: datetime.strptime(x, '%Y-%m-%d'))
data = data.set_index('date')

# This is a Series
data['Temp']

In [ ]:
data['Temp'].plot()

In [ ]:
series = data['Temp']

groups = series.groupby(pd.Grouper(freq='A'))
years = pd.DataFrame()
for name, group in groups:
  years[name.year] = group.values
years.plot(subplots=True, legend=False)

In [ ]:
series.hist()

In [ ]:
groups = series.groupby(pd.Grouper(freq='A'))
years = pd.DataFrame()
for name, group in groups:
  years[name.year] = group.values
years.boxplot()

In [ ]:
one_year = series['1990']
groups = one_year.groupby(pd.Grouper(freq='M'))
months = pd.concat([pd.DataFrame(x[1].values) for x in groups], axis=1)
months = pd.DataFrame(months)
months.columns = range(1,13)
months.boxplot()

In [ ]:
import matplotlib.pyplot as plt

groups = series.groupby(pd.Grouper(freq='A'))
years = pd.DataFrame()
for name, group in groups:
  years[name.year] = group.values
years = years.T
plt.matshow(years, interpolation=None, aspect='auto')

In [ ]:
years

In [ ]:
!wget https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv

In [ ]:
# Read as a dataframe, convert date entries to the index
data = pd.read_csv('airline-passengers.csv',
                   dtype={'Month': str, 'Passengers': int})
data['date'] = data['Month'].apply(\
                                  lambda x: datetime.strptime(x, '%Y-%m'))
data = data.set_index('date')
series = data['Passengers']

series.plot()

In [ ]:
# Rolling statistics, computed over windows -- more or less exactly
# like a sliding window of length 12, stride 1
rolling_mean = pd.Series.rolling(series,center=False,window=12).mean()
rolling_std = pd.Series.rolling(series,center=False,window=12).std()

plt.plot(series,color='blue',label='Series')
plt.plot(rolling_mean,color='red', label='Rolling Mean')
plt.plot(rolling_std,color='gray',label='Rolling Stdev')
plt.legend()

In [ ]:
groups = series.groupby(pd.Grouper(freq='A'))
years = pd.DataFrame()
for name, group in groups:
  years[name.year] = group.values
years.boxplot()

In [ ]:
!wget http://rcmt2.bo.ingv.it/Italydataset1976-2015.csv

In [ ]:
quakes_df = pd.read_csv('Italydataset1976-2015.csv',header=None,names=['ev_id',\
  'date', 'time_orig', 'time_orig_decimals', 'latitude', 'longitude', 'depth_km', \
  'Mb', 'Ms', 'region', 'source_type', 'bwave_n_stations', 'bwave_n_inv', 'bwave_cutoff', \
  'mwave_n_stations', 'mwave_n_inv', 'mwave_cutoff', 'swave_n_stations', 'swave_n_inv', \
  'swave_cutoff', 'dcouple', 'centroid_time', 'delta_centroid_time', 'centroid_lat', \
  'delta_centroid_lat', 'centroid_long', 'delta_centroid_long', 'centroid_depth', \
  'delta_centroid_depth', 'prof', 'half_dur', 'inv_date_string', 'exp', 'mrr', \
  'delta_mrr', 'mss', 'delta_mss', 'mee', 'delta_mee', 'mrs', 'delta_mrs', 'mre', \
  'delta_mre', 'mse', 'delta_mse', 'eigen_t', 'plunge_t', 'strike_t', 'eigen_n', \
  'plunge_n', 'strike_n', 'eigen_p', 'plunge_p', 'strike_p', 'scalar_moment',
  'strike1', 'dip1', 'rake1', 'strike2', 'dip2', 'rake2', 'Mw', 'status_flag', 'quality_flag'])

quakes_df['ts'] = pd.to_datetime(quakes_df.apply(lambda x: datetime.strptime(x['date'] + ' ' +
                                                              x['time_orig'],'%Y-%m-%d %H:%M:%S'), axis=1))

In [ ]:
quakes = quakes_df.set_index('ts')[['latitude','longitude','depth_km','Mw']].rename(columns={'Mw':'magnitude'})
quakes

In the latest Pandas version, we need to query by the index.year, index.month, etc.

In [ ]:
import numpy as np

quakes[quakes.index.year == 2015]['magnitude'].plot()

In [ ]:
plt.hist(quakes[quakes.index.year == 2015]['magnitude'], bins=8)

### Resampling Timeseries

In [ ]:
import pandas as pd
from datetime import datetime

# Read as a dataframe, convert date entries to the index
data = pd.read_csv('daily-min-temperatures.csv', \
                   dtype={'Date': str, 'Temp': float})
data['date'] = data['Date'].apply(\
                                  lambda x: datetime.strptime(x, '%Y-%m-%d'))
data = data.set_index('date')

series = data['Temp']
series[(series.index.year==1990) & (series.index.month==12)].plot()

In [ ]:
series[(series.index.year==1990) & (series.index.month==12)].resample('W').mean().plot()

In [ ]:
series[(series.index.year==1990) & (series.index.month==12)].resample('H').interpolate(method='linear').plot()

In [ ]:
import matplotlib.pyplot as plt

series[(series.index.year==1990) & (series.index.month==12)].plot(color='blue',label='Input')
series[(series.index.year==1990) & (series.index.month==12)].resample('H').interpolate(method='linear').plot(color='purple', label='Linear')
series[(series.index.year==1990) & (series.index.month==12)].resample('H').interpolate(method='spline',order=2).plot(color='green', label='Spline')
plt.legend()

In [ ]:
# SciPy includes signal processing functions
import scipy.signal as signal
import numpy as np

# Original signal -- let's set
# sampling points as t
t = np.linspace(0, 1, 100)
# Then evaluate a sine wave at those points
x = np.sin(2 * np.pi * 5 * t)

# Resample to 100 samples
x_resampled = signal.resample(x, 100)

# Plot the original and resampled signals
import matplotlib.pyplot as plt
plt.plot(t, x, label='Original')
plt.plot(np.linspace(0, 1, 100), x_resampled, label='Resampled')
plt.legend()
plt.show()

## Stationarity of Timeseries

In [ ]:
from statsmodels.tsa.stattools import adfuller
from pandas import Series

def test_stationarity(timeseries, lags=None):
  rolmean = Series.rolling(timeseries, center=False, window=12).mean()
  rolstd = Series.rolling(timeseries, center=False, window=12).std()

  orig = plt.plot(timeseries, color='blue', label='Original')
  mean = plt.plot(rolmean, color='red', label='Rolling Mean')
  std = plt.plot(rolstd, color='black', label='Rolling Std')
  plt.legend(loc='best')
  plt.title('Rolling Mean & Standard Deviation')
  plt.show(block=False)

  print('Results of ADF Test:')
  if lags:
    dftest = adfuller(timeseries, autolag=None, maxlag=lags)
  else:
    dftest = adfuller(timeseries, autolag='AIC')

  dfoutput = pd.Series(dftest[0:4], index=['Test Statistic', 'p-value', '#Lags', 'Observations'])

  for key,value in dftest[4].items():
    dfoutput['Critical Value (%s)'%key] = value
  print(dfoutput)

In [ ]:
# Defaults to 20 lags, not enough to "see" annual seasonality
test_stationarity(data['Temp'])

In [ ]:
# Now give 365 days...
test_stationarity(data['Temp'], 365)

## Timeseries Decomposition and ARIMA

Let's figure out how to decompose timeseries.

Portions of this example are based on https://medium.com/@tirthamutha/time-series-forecasting-using-sarima-in-python-8b75cd3366f2 and https://www.statsmodels.org/dev/examples/notebooks/generated/statespace_sarimax_stata.html#ARIMA-Example-2:-Arima-with-additive-seasonal-effects.

In [ ]:
!wget https://storage.googleapis.com/penn-cis5450/DailyDelhiClimate.csv

Rather than the daily Australian dataset, we'll use a Kaggle dataset about daily temperatures in Delhi.  We know temperatures should be correlated day-to-day and season-to-season -- with a 12 month gap between seasons.

https://www.kaggle.com/datasets/sukhmandeepsinghbrar/daily-delhi-climate

In [ ]:
import pandas as pd

temps = pd.read_csv('DailyDelhiClimate.csv', header=0, index_col=0)

temps = temps[['Temp']]
temps.plot()

Let's look at the correlation between the lags and the data (autocorrelation) for a good region (more than a year)...

In [ ]:
from pandas.plotting import autocorrelation_plot

autocorrelation_plot(temps['Temp'])

And more locally as well...

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

_ = plot_acf(temps)

In [ ]:
plot_acf(temps, lags=1500)

Autocorrelation (as above) considers the correlation of the function's value with its distant previous values, indirectly considering all values in between. We can also look at *partial* autocorrelation, which is a direct correlation between the value and its past lag.

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

_ = plot_pacf(temps)

Here we can see that the 3rd lag, for instance, is less correlated than the 4th, 5th, etc.

## Finding ARIMA/SARIMA Hyperparameters

We have both ARIMA (which follows *small, short-term seasonality within the lags*, as well as longer tremds) and SARIMA (which allows us to include a longer-term seasonal element).

The above plots let us understand how many lags make sense.  We can further fine-tune (p,d,q) for "local" and "seasonal" components with grid search -- but even basic values are likely to work well.

For this step you'll need to restart your kernel, since we are using a library that requires an old version of NumPy.

In [ ]:
!pip install "numpy<2" pmdarima

In [ ]:
import pandas as pd

temps = pd.read_csv('DailyDelhiClimate.csv', header=0, index_col=0)

temps = temps[['Temp']]

import pmdarima as pmd
model = pmd.auto_arima(temps['Temp'],start_p=1,start_q=1,test='adf',m=12,seasonal=True,trace=True)

In [ ]:
temps2 = temps.copy()
temps2.index = pd.DatetimeIndex(temps.index).to_period('D')

In [ ]:
import statsmodels.api as sm

sarima=sm.tsa.statespace.SARIMAX(temps2,order=(1,1,1),seasonal_order=(1,0,1,12))
predicted=sarima.fit().predict();

# Let's compare against a model without the Seasonal component
sarima=sm.tsa.statespace.SARIMAX(temps2,order=(1,1,1))
predicted2=sarima.fit().predict();

In [ ]:
import matplotlib.pyplot as plt

predicted.index = temps.index

plt.figure(figsize=(20,6))
plt.plot(temps['Temp'],label='Actual')
plt.plot(predicted,label='Predicted')
# plt.plot(predicted,label='Nonseasonal')

plt.legend()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

df = temps.copy()

In [ ]:
result = seasonal_decompose(df['Temp'], model='multiplicative', period=12)
trend = result.trend.dropna()
seasonal = result.seasonal.dropna()
residual = result.resid.dropna()

# Plot the decomposed components
plt.figure(figsize=(4,6))

plt.subplot(4, 1, 1)
plt.plot(df['Temp'], label='Original Series')
plt.legend()

plt.subplot(4, 1, 2)
plt.plot(trend, label='Trend')
plt.legend()

plt.subplot(4, 1, 3)
plt.plot(seasonal, label='Seasonal')
plt.legend()

plt.subplot(4, 1, 4)
plt.plot(residual, label='Residuals')
plt.legend()

plt.tight_layout()
plt.show()


## Exercises

Let's try to use ARIMA on salary data.

## Autograder setup

In [ ]:
#PLEASE ENSURE YOUR PENN-ID IS ENTERED CORRECTLY. IF NOT, THE AUTOGRADER WON'T KNOW WHO
#TO ASSIGN POINTS TO YOU IN OUR BACKEND
STUDENT_ID = 99999999 # YOUR PENN-ID GOES HERE AS AN INTEGER##PLEASE ENSURE YOUR PENN-ID IS ENTERED CORRECTLY. IF NOT, THE AUTOGRADER WON'T KNOW WHO

In [ ]:
%%writefile notebook-config.yaml

grader_api_url: 'https://23whrwph9h.execute-api.us-east-1.amazonaws.com/default/Grader23'
grader_api_key: 'flfkE736fA6Z8GxMDJe2q8Kfk8UDqjsG3GVqOFOa'

In [ ]:
%set_env HW_ID=cis5450_25f_HW9

In [ ]:
!pip3 install penngrader-client

In [ ]:
import os
from penngrader.grader import *

grader = PennGrader('notebook-config.yaml', os.environ['HW_ID'], STUDENT_ID, STUDENT_ID)

## Data

In [ ]:
!wget -nc https://storage.googleapis.com/penn-cis5450/Month_Value_1.csv

In [ ]:
payroll_sales_df = pd.read_csv('Month_Value_1.csv')

display(payroll_sales_df)

payroll_sales_df.describe()

Let's convert the date.

In [ ]:
from datetime import datetime
from statsmodels.tsa.stattools import adfuller

payroll_sales_df['date'] = payroll_sales_df['Period'].apply(\
                                  lambda x: datetime.strptime(x, '%m.%d.%Y'))
payroll_sales_df = payroll_sales_df.set_index('date')
payroll_sales_df.drop(columns=['Period'], inplace=True)

payroll_sales_df.plot()

There are some null values, let's fill them in via k-nearest-neighbor.

In [ ]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)
values = ['Revenue','Sales_quantity','Average_cost','The_average_annual_payroll_of_the_region']

for column in values:
  result = imputer.fit_transform      # TODO
  payroll_sales_df[column] = result.reshape(-1,1)

Compute the ADF over the average cost.

In [ ]:
statistic = # TODO

In [ ]:
statistic

In [ ]:
grader.grade(test_case_id = 'test_statistic', answer = statistic)